# Firewatch SFT Training - Kaggle

Thin launcher for the real Hub-backed SFT pipeline. Durable artifacts are written to Hugging Face Hub model repos, not to the Kaggle VM.

## 1. Runtime Setup

Enable GPU acceleration and add `HF_TOKEN` in Kaggle Secrets before running this notebook.

In [ ]:
!nvidia-smi
!pip install -q uv

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Set HF_TOKEN in Kaggle Secrets before running training.')
os.environ['HF_TOKEN'] = token
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

## 2. Repository Setup

Clone the project into `/kaggle/working`. Update `REPO_URL` for your working branch or fork.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/10doshi12/sst_hack_openenv.git'
repo_dir = Path('/kaggle/working/sst_hack_openenv')
if not repo_dir.exists():
    !git clone {REPO_URL} {repo_dir}
%cd /kaggle/working/sst_hack_openenv/firewatch_agent
!uv sync
!uv pip install -e '.[fast]'
!uv pip install -q "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"

## 3. Preflight Then Train

Preflight must pass before the base LLM is loaded. Install Unsloth with ``uv pip`` after ``uv sync`` so ``uv run`` sees it.

In [ ]:
!uv run python -m sft.preflight --config config.yaml
!uv run python -m sft.train --config config.yaml

Expected outputs:
- `<namespace>/firewatch-gnn/gnn/batch_NNN.pt`
- `<namespace>/firewatch-gnn/gnn/normalization.json`
- `<namespace>/firewatch-agent-sft/batch_NNN/`